← [P1 · What's out there](p1_whats_out_there.ipynb) · [Index](../README.md) · [P3 · How we observe](p3_how_we_observe.ipynb) →
<!--nav-->

# P2 · Stars

Everything in this repo is a measurement of a star. The planet is inferred from what it
does to the starlight, so the star sets your noise, your reference frame, and the meaning
of every number you extract. "Depth 8,400 ppm" only becomes "a Jupiter-sized planet" if you
know how big the star is.

**You'll learn:** what temperature does to colour · the spectral sequence OBAFGKM · how to
read an HR diagram (and build one from real data) · why small cool stars are the best
transit targets · and how stars end.

## 1. A star is a blackbody, roughly

A hot dense object emits light across all wavelengths with a shape set almost entirely by
its **temperature**. Two consequences that come up constantly:

- **Hotter = bluer.** The peak wavelength shifts as 1/T (Wien's law).
- **Hotter = vastly brighter per unit area.** Total emission goes as T⁴
  (Stefan–Boltzmann), so a 2× hotter star radiates 16× more from each square metre.

That's why "colour" is a proxy for temperature, and why colour is cheap to measure — two
filters and a ratio, no spectrograph required.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from astropy import units as u
from astropy.modeling.physical_models import BlackBody

from skyplay import plotting

plotting.use_style()

wavelength = np.linspace(100, 3000, 600) * u.nm
STARS = [('M dwarf (3,200 K)', 3200), ('Sun, G type (5,800 K)', 5800), ('A type (9,500 K)', 9500)]

fig, ax = plt.subplots(figsize=(9, 4.5))
for (label, temp), colour in zip(STARS, plotting.SERIES):
    flux = BlackBody(temperature=temp * u.K)(wavelength)
    ax.plot(wavelength, flux / flux.max(), label=label, color=colour)
    ax.axvline(2.898e6 / temp, color=colour, ls=':', lw=1)   # Wien peak, nm

ax.axvspan(380, 750, color=plotting.INK['grid'], alpha=0.5, zorder=0)
ax.annotate('visible', xy=(560, 1.03), ha='center', color=plotting.INK['secondary'], fontsize=9)
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Emission (each curve scaled to its own peak)')
ax.set_title('Same shape, shifted by temperature — dotted lines mark the peak')
ax.legend()
plt.show()

print('Scaled to their own peaks so the shift is visible. In absolute terms the A star')
print('outshines the M dwarf per unit area by (9500/3200)**4 =', round((9500/3200)**4), 'times.')

## 2. The spectral sequence: OBAFGKM

Stars were classified by their spectra before anyone knew why, and the ordering turned out
to be a temperature sequence. Hottest to coolest:

| Class | Temperature | Colour | Mass (Suns) | Main-sequence life | Share of stars |
|---|---|---|---|---|---|
| **O** | 30,000–50,000 K | blue | 16–100 | a few Myr | ~0.00003% |
| **B** | 10,000–30,000 K | blue-white | 2–16 | ~100 Myr | 0.1% |
| **A** | 7,500–10,000 K | white | 1.4–2 | ~1 Gyr | 0.6% |
| **F** | 6,000–7,500 K | yellow-white | 1.0–1.4 | ~3 Gyr | 3% |
| **G** | 5,200–6,000 K | yellow (the Sun) | 0.8–1.0 | ~10 Gyr | 8% |
| **K** | 3,700–5,200 K | orange | 0.45–0.8 | ~30 Gyr | 12% |
| **M** | 2,400–3,700 K | red | 0.08–0.45 | >100 Gyr | **~76%** |

Three things worth extracting from that table:

**M dwarfs are the universe's default star.** Three quarters of all stars are M dwarfs.
Every intuition built on the Sun is a minority case.

**Lifetime runs opposite to mass.** Massive stars have far more fuel but burn it
extravagantly — luminosity scales roughly as mass³·⁵, so lifetime scales as ~M/M³·⁵ =
M⁻²·⁵. An O star lives a few million years; an M dwarf will outlive the current age of the
universe many times over.

**M dwarfs are the best transit targets**, for two independent reasons:

- Depth is (Rp/Rs)². A small star means a *deeper* transit for the same planet.
- They're cool, so their habitable zone is close in, so interesting planets have short
  periods — and short periods mean more transits per observing campaign.

That is why the target-selection advice in the main README points at M dwarfs.

In [ ]:
masses = np.array([0.1, 0.3, 0.5, 1.0, 2.0, 5.0, 16.0])

# Rough main-sequence scalings: L ~ M**3.5, so lifetime ~ M / L ~ M**-2.5
luminosity = masses ** 3.5
lifetime_gyr = 10.0 * masses ** -2.5     # normalised to the Sun's ~10 Gyr

print(f'{"mass (Msun)":>12s} {"luminosity (Lsun)":>18s} {"MS lifetime":>16s}')
for m, lum, life in zip(masses, luminosity, lifetime_gyr):
    span = f'{life * 1000:.0f} Myr' if life < 1 else f'{life:.0f} Gyr'
    print(f'{m:12.2f} {lum:18.3f} {span:>16s}')

print('\n-> A 16 Msun star is ~16,000x brighter than the Sun and lives ~0.1% as long.')
print('   A 0.1 Msun M dwarf is ~3,000x fainter and effectively immortal.')

## 3. The HR diagram, from real data

Plot luminosity against temperature for a lot of stars and they do not scatter randomly —
they fall on tracks. That plot is the **Hertzsprung–Russell diagram**, and it's arguably
the single most important diagram in astronomy: it's stellar evolution made visible.

Let's build one from **Gaia** data. Gaia measured positions, motions and brightnesses for
~2 billion stars, and crucially it measured **parallax** — so we can convert apparent
brightness to true luminosity.

Absolute magnitude from apparent magnitude and parallax:

```
M_G = G + 5·log₁₀(parallax_in_mas) − 10
```

Colour `BP−RP` stands in for temperature — bluer (smaller) is hotter, so the hot end is on
the left.

In [ ]:
from astroquery.gaia import Gaia

# Nearby stars with well-measured parallaxes and colours. ~15 s.
job = Gaia.launch_job('''
    SELECT TOP 8000 phot_g_mean_mag, bp_rp, parallax
    FROM gaiadr3.gaia_source
    WHERE parallax > 10
      AND parallax_over_error > 20
      AND phot_g_mean_mag IS NOT NULL
      AND bp_rp IS NOT NULL
      AND phot_bp_mean_flux_over_error > 20
''')
stars = job.get_results()
print(f'{len(stars)} stars within {1000 / 10:.0f} pc with good parallaxes')

g = np.asarray(stars['phot_g_mean_mag'], dtype=float)
colour = np.asarray(stars['bp_rp'], dtype=float)
parallax = np.asarray(stars['parallax'], dtype=float)
abs_g = g + 5 * np.log10(parallax) - 10

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 7))
ax.scatter(colour, abs_g, s=3, alpha=0.35, color=plotting.SERIES[0], linewidths=0,
           rasterized=True)

ax.invert_yaxis()          # brighter upward, as tradition demands
ax.set_xlabel('Colour  BP - RP   (bluer/hotter <-- --> redder/cooler)')
ax.set_ylabel('Absolute magnitude $M_G$  (brighter upward)')
ax.set_title('Hertzsprung-Russell diagram, Gaia DR3')

ax.annotate('main sequence\n(hydrogen burning)', xy=(1.5, 7.0), xytext=(2.1, 3.0),
            color=plotting.INK['primary'], fontsize=9,
            arrowprops=dict(arrowstyle='->', color=plotting.INK['secondary'], lw=1))
ax.annotate('white dwarfs\n(dead cores)', xy=(0.5, 13.0), xytext=(-0.4, 15.6),
            color=plotting.INK['primary'], fontsize=9,
            arrowprops=dict(arrowstyle='->', color=plotting.INK['secondary'], lw=1))
ax.annotate('the Sun sits here', xy=(0.82, 4.67), xytext=(-0.3, 1.2),
            color=plotting.INK['primary'], fontsize=9,
            arrowprops=dict(arrowstyle='->', color=plotting.INK['secondary'], lw=1))
plt.show()

Read that diagram carefully, because a lot follows from it:

- **The diagonal band is the main sequence** — stars fusing hydrogen in their cores, which
  is where a star spends ~90% of its life. Position along it is set almost entirely by
  **mass**.
- **The dense red end (bottom right) is M dwarfs.** They dominate by number, exactly as the
  table said.
- **The sparse clump at bottom left is white dwarfs** — hot but tiny, so faint. Dead cores
  of stars like the Sun, cooling forever.
- **Empty regions are physically empty**, not undersampled. Stars cross them so quickly that
  catching one is unlikely.

This sample is deliberately *nearby* stars, which biases it: intrinsically faint M dwarfs are
only detectable close by, so they're over-represented relative to a distant sample, and rare
giants are largely missing. **Every catalogue has a selection function** — the same lesson as
notebook 06.

## 4. How stars end

When core fusion stops, gravity wins. What's left depends only on mass:

| Initial mass | Endpoint | What it is |
|---|---|---|
| < 0.08 M☉ | Brown dwarf | Never fused hydrogen at all |
| 0.08–8 M☉ | **White dwarf** | Earth-sized carbon/oxygen core, held up by electron degeneracy. The Sun's fate |
| 8–25 M☉ | **Neutron star** | ~1.4 M☉ inside ~20 km. A sugar cube weighs ~100 million tonnes. Seen as pulsars |
| > 25 M☉ | **Black hole** | Collapse continues without limit |

These remnants matter for observation far beyond their size:

- **White dwarfs** accreting from a companion produce **novae**, and if pushed past ~1.4 M☉
  they detonate as **Type Ia supernovae** — the standardisable candles used to discover
  dark energy.
- **Neutron stars** in binaries give X-ray binaries and millisecond pulsars; the merger of
  two produces a **kilonova**, forging heavy elements.
- **Black holes** feeding at galactic centres are **AGN/quasars**; merging stellar-mass ones
  are what LIGO hears.

Notice how many of those require a **binary**. Roughly half of all stars have a companion,
and binary interaction is what turns quiet stellar corpses into the most energetic events in
the universe. Which is worth remembering the next time notebook 04 calls an eclipsing binary
an "impostor" — that framing is about *our* goal, not their importance.

**Next:** [`p3_how_we_observe.ipynb`](p3_how_we_observe.ipynb) — the instruments, and where the noise comes from.